In [66]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, average_precision_score,accuracy_score,confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.pipeline import Pipeline
import numpy as np
import joblib

In [41]:
train = pd.read_csv("D:/NamCuoi/Code/Project/PhatHienNoiDungDocHai/Data/ViCTSD-main/ViCTSD_train.csv")
val = pd.read_csv("D:/NamCuoi/Code/Project/PhatHienNoiDungDocHai/Data/ViCTSD-main/ViCTSD_valid.csv")

print("Train shape:",train.shape)
print("Val shape:",val.shape)
print("Thong tin Train:")
print(train.info())
train.head()

Train shape: (7000, 6)
Val shape: (2000, 6)
Thong tin Train:
<class 'pandas.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Unnamed: 0        7000 non-null   int64
 1   Comment           7000 non-null   str  
 2   Constructiveness  7000 non-null   int64
 3   Toxicity          7000 non-null   int64
 4   Title             7000 non-null   str  
 5   Topic             7000 non-null   str  
dtypes: int64(3), str(3)
memory usage: 328.2 KB
None


,Unnamed: 0,Comment,Constructiveness,Toxicity,Title,Topic
0,6326,Thật tuyệt vời...!!!,0,0,Những 'bước tiến diệu kỳ' của Trúc Nhi - Diệu Nhi,SucKhoe
1,7835,"mỹ đã tuột dốc quá nhiều rồi, giờ muốn vực dậy...",1,0,Hình tượng Mỹ sụp đổ trong lòng người dân thế ...,TheGioi
2,4690,tôi thấy người lái xe hơi bấm còi mới là người...,1,1,Cả trăm người đạp xe thể dục bịt kín đường,OtoXemay
3,6011,Coi dịch là giặc. Đã mang tên đó mà xâm nhập V...,0,0,11 ngày không lây nhiễm nCoV cộng đồng,SucKhoe
4,9303,Thương các bé quá! Các con còn quá nhỏ mà đã p...,0,0,5 trẻ chết đuối dưới ao,ThoiSu


In [42]:
print(train["Toxicity"].value_counts())
print(train["Toxicity"].value_counts(normalize=True))

Toxicity
0    6241
1     759
Name: count, dtype: int64
Toxicity
0    0.891571
1    0.108429
Name: proportion, dtype: float64


In [47]:
print(val["Toxicity"].value_counts())
print(val["Toxicity"].value_counts(normalize=True))

Toxicity
0    1768
1     232
Name: count, dtype: int64
Toxicity
0    0.884
1    0.116
Name: proportion, dtype: float64


In [43]:
def prepare_data(df):
    df = df.dropna(subset=["Comment", "Toxicity"]).copy()

    X = df["Comment"]
    y = df["Toxicity"]
    return X,y

In [44]:
X_train, y_train = prepare_data(train)
X_val, y_val = prepare_data(val)

In [71]:
#Xây dựng popeline baseline
baseline = Pipeline([
    ("tfidf",TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_features=100000)),
    ("clf",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42))
])
#train
baseline.fit(X_train,y_train)
#đánh giá trên val
val_pred = baseline.predict(X_val)

print("Val accuracy:",accuracy_score(y_val,val_pred))
print("Val Macro-F1:",f1_score(y_val,val_pred,average="macro"))

print(classification_report(y_val,val_pred,labels=[0,1],target_names=["Khong doc hai","Doc hai"],zero_division=0))
print("Validation Confusion Matrix(Ma trận nhần lẫn trên tập val):")
print(confusion_matrix(y_val, val_pred, labels=[0, 1]))

Val accuracy: 0.856
Val Macro-F1: 0.6887906274110622
               precision    recall  f1-score   support

Khong doc hai       0.94      0.90      0.92      1768
      Doc hai       0.41      0.53      0.46       232

     accuracy                           0.86      2000
    macro avg       0.67      0.71      0.69      2000
 weighted avg       0.87      0.86      0.86      2000

Validation Confusion Matrix(Ma trận nhần lẫn trên tập val):
[[1589  179]
 [ 109  123]]


In [60]:
# Tìm mẫu bị dự đoán sai trong val
error_df = pd.DataFrame({"Comment":X_val,"True_Label":y_val,"Predicted_Label":val_pred})
error_df = error_df[ error_df["True_Label"] != error_df["Predicted_Label"] ]
print("Mẫu dự đoán sai:",len(error_df))
print(error_df.head(10).to_string())

Mẫu dự đoán sai: 288
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      Comment  True_Label  Predicted_Label
2                                                                                                                                                                                                                                                                                                                                                                                                                                      dám chui ra,vậy ăn vô lại ếch said :')           0               

In [70]:
#Xây dựng popeline baseline
baseline = Pipeline([
    ("tfidf",TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_features=100000)),
    ("svm",LinearSVC(C=1.0,random_state=42))
])
#train
baseline.fit(X_train,y_train)
#đánh giá trên val
val_pred = baseline.predict(X_val)

print("Val accuracy:",accuracy_score(y_val,val_pred))
print("Val Macro-F1:",f1_score(y_val,val_pred,average="macro"))

print(classification_report(y_val,val_pred,labels=[0,1],target_names=["Khong doc hai","Doc hai"],zero_division=0))
print("Ma trận nhầm lẫn trên tập val")
print(confusion_matrix(y_val, val_pred, labels=[0, 1]))

Val accuracy: 0.8975
Val Macro-F1: 0.646787706489199
               precision    recall  f1-score   support

Khong doc hai       0.91      0.98      0.94      1768
      Doc hai       0.66      0.24      0.35       232

     accuracy                           0.90      2000
    macro avg       0.79      0.61      0.65      2000
 weighted avg       0.88      0.90      0.88      2000

Ma trận nhầm lẫn trên tập val
[[1740   28]
 [ 177   55]]
